# Designer Decision Explorer

**Post-result exploration of EnergyPlus optimization results — for the building designer.**

Raw simulation outputs (kWh, JSON, Pareto sets) are meaningful to the *simulationist*, not the *designer*. Following Bleil de Souza & Tucker (2013), *Thermal Simulation Software Outputs: What Do Building Designers Propose?* (BS2013 / IBPSA), this notebook re-frames already-computed results around the designer's **Inferred Goals**:

| Inferred Goal | Question it answers |
|---|---|
| Optimizing | Which designs are the best trade-offs? |
| Understanding a result | Where does a design sit, and what are its parameters? |
| Which variable is responsible | Which design parameter drives performance? |
| Meeting a target | How far is a design from a benchmark? *(stub)* |

This notebook is **post-result**: run `/simulate` (mode `pareto` or `exhaustive`) first, then explore here. All figures are reused from `scripts/dashboard.py` — no new plotting code.

> Part of the `/design` process — see `.claude/commands/design.md`.

In [ ]:
# Step 0 — setup. Kernel must be _GCP_VM_VERSION/.venv with plotly installed
# (setup_gcp_env.sh --with-design).
import sys, json
from pathlib import Path

sys.path.insert(0, 'scripts')  # dashboard.py lives in _GCP_VM_VERSION/scripts/
from dashboard import _normalize_pareto, load_data, fig_pareto_front, fig_pareto_tradeoff, fig_gsa_morris

import plotly.io as pio
pio.renderers.default = 'notebook'

print('Designer Decision Explorer — figures ready (reused from scripts/dashboard.py).')

## Step 0 — Load results

Point the paths below at your result JSONs. The defaults look for a VM run staged under `/tmp/energyplus_sim/` and the committed GSA results in `plan/`. If nothing is found, the notebook falls back to the embedded 2026-05-18 demo run so every cell still renders.

In [ ]:
# Designer: edit these if your result files are elsewhere.
PARETO_CANDIDATES = [
    '/tmp/energyplus_sim/exhaustive/exhaustive_pareto.json',  # /simulate exhaustive (M3.7)
    '/tmp/energyplus_sim/pareto/pareto_front.json',           # /simulate pareto
]
GSA_CANDIDATES = [
    '/tmp/energyplus_sim/gsa/gsa_results.json',
    'data/gsa_results_20260518.json',
]

def _first(paths):
    return next((p for p in paths if Path(p).exists()), None)

def load_results():
    pareto_path, gsa_path = _first(PARETO_CANDIDATES), _first(GSA_CANDIDATES)
    if pareto_path:
        raw = json.loads(Path(pareto_path).read_text(encoding='utf-8'))
        # exhaustive_pareto.json uses all_results; dashboard figures expect all_candidates
        if 'all_results' in raw and 'all_candidates' not in raw:
            raw['all_candidates'] = raw['all_results']
        pareto = _normalize_pareto(raw)
        print(f'Pareto: {pareto_path}')
    else:
        pareto, _ = load_data(None, None)
        print('Pareto: not found — using embedded demo data (2026-05-18 run)')
    if gsa_path:
        gsa = json.loads(Path(gsa_path).read_text(encoding='utf-8'))
        print(f'GSA   : {gsa_path}')
    else:
        _, gsa = load_data(None, None)
        print('GSA   : not found — using Morris N=15 fallback')
    return pareto, gsa

pareto, gsa = load_results()
n_front = len(pareto.get('pareto_front', []))
print(f'Non-dominated designs: {n_front}')

## Inferred Goal — *Optimizing*

*Bleil de Souza & Tucker (2013), Table 1: find the optimum quantities for a specific set of parameters to achieve a best performance target.*

The **Pareto front** is the set of non-dominated designs: no other design is better on *both* comfort (FR1) and energy (FR2). It does not pick one answer — it shows the designer the room for trade-off.

In [ ]:
fig_pareto_front(pareto).show()

## Inferred Goal — *Understanding a specific performance result*

*Understanding where a performance result is happening and what building elements are responsible for causing it.*

Pick a non-dominated design (`SELECTED` below) and read its design parameters — the actual envelope / operation choices behind the number.

In [ ]:
import pandas as pd

front = pareto.get('pareto_front', [])
SELECTED = 0  # designer: change to inspect another non-dominated design

if front:
    c = front[SELECTED]
    row = {
        'candidate': c['candidate_id'],
        'orientation': c.get('orient_label', c.get('orientation')),
        'setpoint (C)': c.get('setpoint'),
        'wall R': c.get('wall_r'),
        'roof R': c.get('roof_r'),
        'HVAC energy (kWh)': c.get('hvac_energy_kwh'),
    }
    display(pd.DataFrame([row]))
    fig_pareto_tradeoff(pareto).show()
else:
    print('No Pareto front in the loaded data.')

## Inferred Goal — *Which design variable is responsible?*

The paper's Table 2 gives the **Tornado Diagram** as the representation for *the contribution of each design parameter*.

Global Sensitivity Analysis (Morris **μ***) ranks the design parameters by influence. **σ** flags non-linear / interaction effects — a parameter whose effect depends on the others.

In [ ]:
fig_gsa_morris(gsa).show()

## Inferred Goal — *Meeting a target*  ⚠️ *stub*

*Quantify how far a performance result is from a prescribed benchmark, and inform which design variables are responsible for the mismatch.*

This section is an **initial stub**. It reports the gap to the base case; the *which variable is responsible for the mismatch* part is left for the next iteration (it needs a benchmark-aware analysis not yet in `dashboard.py`).

In [ ]:
# STUB — gap to the base case. Benchmark: /simulate validate base run.
BENCHMARK_KWH = 64519.5

front = pareto.get('pareto_front', [])
if front:
    best = min(front, key=lambda c: c['hvac_energy_kwh'])
    gap = best['hvac_energy_kwh'] - BENCHMARK_KWH
    pct = gap / BENCHMARK_KWH * 100
    cid = best['candidate_id']
    energy = best['hvac_energy_kwh']
    print(f'Best non-dominated design : {energy:>10.0f} kWh  ({cid})')
    print(f'Benchmark (base case)     : {BENCHMARK_KWH:>10.0f} kWh')
    print(f'Gap                       : {gap:>+10.0f} kWh  ({pct:+.1f}%)')
else:
    print('No Pareto front in the loaded data.')

## Where this fits — the framework

Bleil de Souza & Tucker (2013) propose 5 framework parts. This notebook is an early instance of parts 1, 3 and 4:

1. **Inferred Goals** — the section headings above.
2. **Design Actions** — *not yet* (a live design loop is out of scope for this post-result notebook).
3. **Analysis Processes** — the reused `dashboard.py` figures (comparative post-processing).
4. **Metrics & representation systems** — Pareto plot, trade-off band, Morris / Tornado bars.
5. **Patterns for Decision Making** — *not yet*.

### Stubs / next steps
- *Meeting a target* — add a benchmark-aware *which DP causes the mismatch* analysis.
- Location-based representation (floor plans) once the model carries real plan geometry.

See `.claude/commands/design.md` for the full `/design` process.

## Step — Export & share

Generate a standalone HTML report and stage it (with this notebook) to the shared GCS
bucket under `data/gcp_vm_design_<timestamp>/`, so the whole team can open the same
results without re-running anything.

**Save the notebook first (Ctrl+S)** so the staged `.ipynb` includes your rendered figures.

In [ ]:
from datetime import datetime, timezone
from google.cloud import storage
from dashboard import save_static

BUCKET = 'eplus-colab-cloud-data'
ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
prefix = f'data/gcp_vm_design_{ts}'

# Standalone HTML report (reuses the dashboard 4-panel layout)
report = f'/tmp/design_report_{ts}.html'
save_static(pareto, gsa, report)

# Stage the report + this notebook to the shared bucket
client = storage.Client()
bkt = client.bucket(BUCKET)
artifacts = [
    (report, 'design_report.html'),
    ('Designer_Decision_Explorer.ipynb', 'Designer_Decision_Explorer.ipynb'),
]
for local, name in artifacts:
    bkt.blob(f'{prefix}/{name}').upload_from_filename(local)
    print(f'staged -> gs://{BUCKET}/{prefix}/{name}')

print('\nShare this prefix with the team:')
print(f'  gs://{BUCKET}/{prefix}/')